# Semantic Caching for RAG with Qdrant

A RAG pipeline answers a question in two steps: it retrieves matching documents from a knowledge base, then an LLM writes the answer from them. A semantic cache stores past questions with their answers in Qdrant. When a new question is close enough in meaning to a cached one, the cache serves the stored answer and skips both steps, saving time and LLM cost.

This notebook builds such a cache, calibrates the similarity threshold that defines "close enough" on a labeled set of questions, adds an optional second check with the Jev model, and keeps cached answers fresh. It accompanies the [Semantic Caching for RAG](https://qdrant.tech/documentation/tutorials-build-essentials/semantic-cache/) tutorial.

You need a free [Qdrant Cloud](https://cloud.qdrant.io/) cluster and an [OpenRouter](https://openrouter.ai/) API key.

## Setup

In [1]:
%pip install -q --disable-pip-version-check "qdrant-client>=1.19.1" openrouter

Note: you may need to restart the kernel to use updated packages.

In [2]:
import os
from getpass import getpass

for variable in ("QDRANT_URL", "QDRANT_API_KEY", "OPENROUTER_API_KEY"):
    if not os.getenv(variable):
        os.environ[variable] = getpass(f"{variable}: ")

In [3]:
import os

from openrouter import OpenRouter
from qdrant_client import QdrantClient, models

client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True,
    # An external embedding provider, such as OpenRouter, can take longer than the default 5 seconds.
    timeout=60,
)

openrouter = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"])

The collection names, the embedding model, the LLM for RAG, and an embedding helper for Qdrant Cloud Inference.

In [4]:
# Cloud Inference forwards "openrouter/..." models to OpenRouter with the key passed in options.
EMBEDDING_MODEL = "openrouter/openai/text-embedding-3-small"
EMBEDDING_SIZE = 1536
LLM_MODEL = "openai/gpt-6-luna"

KNOWLEDGE_BASE_COLLECTION = "semcache_tut_kb"
CACHE_COLLECTION = "semcache_tut_cache"


def embed(text: str) -> models.Document:
    # Qdrant Cloud turns text into a vector on the server.
    return models.Document(
        text=text,
        model=EMBEDDING_MODEL,
        options={"openrouter-api-key": os.environ["OPENROUTER_API_KEY"]},
    )

## Build the RAG (before the Cache)

The knowledge base: short policy documents of a marketplace.

In [5]:
from typing import TypedDict


class Doc(TypedDict):
    id: str
    text: str


DOCUMENTS: list[Doc] = [
    {
        "id": "returns-v1",
        "text": "You can return most items within 30 days of delivery for a refund to the original payment method. Items must be unworn and unwashed, with the original tags attached. Shoes may be tried on indoors, but shoes with any outdoor wear on the soles cannot be returned. Sale items can be returned within 30 days for store credit only, not a refund. Items marked Final Sale, gift cards, and opened packs of socks cannot be returned. Exchanges for a different size or color follow the same 30-day and condition rules and are free. For refunds, a $6.95 return shipping fee is deducted from the refund. Refunds are issued within 5 business days after the return reaches our warehouse.",
    },
    {
        "id": "shipping-v1",
        "text": "We ship from our warehouse in Ohio to the United States, Canada, and the European Union. We do not ship to PO boxes or other countries. Standard shipping within the US takes 3 to 5 business days and is free on orders over $75; orders of $75 or less pay $5.95. US express shipping takes 1 to 2 business days and costs $14.95. Shipping to Canada takes 5 to 8 business days and costs $12.95. Shipping to the European Union takes 7 to 12 business days and costs $19.95, and the customer pays import duties on delivery. Orders placed before 2 p.m. Eastern Time on a business day ship the same day; later orders ship the next business day.",
    },
    {
        "id": "payment-v1",
        "text": "We accept Visa, Mastercard, American Express, PayPal, and Fernhollow gift cards. We do not accept cash on delivery, checks, or cryptocurrency. Your card is charged when the order ships, not when you place it. Orders between $150 and $1,000 can be paid in 4 interest-free installments at checkout. You can combine up to two gift cards in one order, and a gift card can be combined with one credit card. All prices are in US dollars. Customers in Canada and the European Union are also charged in US dollars, and their bank may add a currency conversion fee. Promo codes must be entered before payment and cannot be applied to orders that were already placed.",
    },
    {
        "id": "warranty-v1",
        "text": "All footwear has a 1-year warranty against manufacturing defects, such as sole separation, broken eyelets, or failed seams. Jackets and other outerwear have a 2-year warranty covering zippers, seams, and waterproof coatings. Backpacks have a 3-year warranty. The warranty period starts on the delivery date. The warranty does not cover normal wear, such as worn-down tread, damage from misuse, or items altered after purchase. To make a claim, email photos of the defect and your order number to warranty@fernhollow.example. Approved claims get a free replacement of the same item; if the item is no longer available, you get store credit for the purchase price. Warranty claims do not require the original tags.",
    },
    {
        "id": "account-v1",
        "text": "You can check out as a guest, but an account lets you see your order history and save addresses. To reset your password, click Forgot password on the sign-in page; the reset link expires after 60 minutes. You can change your email address in Account Settings; the change takes effect after you confirm it from the new address. Guest orders can be added to a new account if you sign up with the same email address within 90 days of the order. To delete your account, contact support; deletion is permanent and takes up to 30 days. Members of the free Trail Club earn 1 point per dollar spent, and 100 points equal a $5 reward.",
    },
    {
        "id": "order-tracking-v1",
        "text": "When your order ships, we email you a tracking number. Account holders also see order status under Orders in their account. For guest orders, use the Track Order page with your order number and email address. Tracking information can take up to 24 hours to appear after the shipping email. You can cancel or change an order within 1 hour of placing it; after that, it cannot be changed. If tracking shows no movement for 5 business days, contact support and we will open a trace with the carrier. If a package is marked delivered but you cannot find it, report it on the Track Order page within 7 days and we will file a claim with the carrier.",
    },
]

In [6]:
def create_knowledge_base(documents: list[Doc]) -> None:
    client.create_collection(
        KNOWLEDGE_BASE_COLLECTION,
        vectors_config=models.VectorParams(
            size=EMBEDDING_SIZE,
            distance=models.Distance.COSINE,
        ),
    )
    client.upsert(
        KNOWLEDGE_BASE_COLLECTION,
        points=[
            models.PointStruct(
                id=i,
                vector=embed(doc["text"]),
                payload={"doc_id": doc["id"], "text": doc["text"]},
            )
            for i, doc in enumerate(documents)
        ],
        wait=True,
    )


create_knowledge_base(DOCUMENTS)

The RAG pipeline: retrieve the two closest documents, then generate the answer. It measures cost and latency, so we can compare it with the cache, and fails with a clear error when the LLM returns no answer.

In [7]:
import time

SYSTEM_PROMPT = (
    "You are a store support assistant. Answer in one or two sentences, "
    "using only the documents provided. If they do not contain the answer, say so."
)


class Answer(TypedDict):
    answer: str
    source_doc_ids: list[str]
    latency_s: float
    cost_usd: float


def rag_answer(question: str) -> Answer:
    start = time.perf_counter()

    hits = client.query_points(
        KNOWLEDGE_BASE_COLLECTION,
        query=embed(question),
        # Each document covers one policy; a second one helps questions that touch two policies.
        limit=2,
    ).points
    context = "\n\n".join(f"[{h.payload['doc_id']}]\n{h.payload['text']}" for h in hits)

    response = openrouter.chat.send(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Documents:\n{context}\n\nQuestion: {question}",
            },
        ],
        # The model spends hidden reasoning tokens before it writes the answer, so leave room for them.
        max_completion_tokens=2000,
    )

    choice = response.choices[0]
    # Fail with a clear error if the answer was cut off or is empty.
    if choice.finish_reason != "stop" or not choice.message.content:
        raise RuntimeError(
            f"The LLM returned no usable answer (finish_reason={choice.finish_reason})."
        )

    return {
        "answer": choice.message.content.strip(),
        "source_doc_ids": [h.payload["doc_id"] for h in hits],
        "latency_s": time.perf_counter() - start,
        "cost_usd": response.usage.cost,
    }

In [8]:
result = rag_answer("Do I pay for return shipping when I return an item for a refund?")
print(result["answer"])
print(
    result["source_doc_ids"],
    f"{result['latency_s']:.2f} s",
    f"${result['cost_usd']:.6f}",
)

Yes. A $6.95 return shipping fee is deducted from your refund.
['returns-v1', 'payment-v1'] 2.69 s $0.000045


## Create the Cache Collection

Each cached question is one point: the question's embedding is the vector; the payload holds the question, the answer, the documents it was built from, and when it expires.

In [9]:
from datetime import timedelta


def create_cache() -> None:
    client.create_collection(
        CACHE_COLLECTION,
        vectors_config=models.VectorParams(
            size=EMBEDDING_SIZE,
            distance=models.Distance.COSINE,
        ),
    )
    # New Qdrant Cloud collections run in strict mode: filtering on a field without a payload index is rejected.
    client.create_payload_index(
        CACHE_COLLECTION, "source_doc_ids", models.PayloadSchemaType.KEYWORD, wait=True
    )
    client.create_payload_index(
        CACHE_COLLECTION, "expires_at", models.PayloadSchemaType.DATETIME, wait=True
    )


# Optional: how long a cached answer stays valid.
CACHE_TTL = timedelta(days=7)

create_cache()

In [10]:
SEED_QUESTIONS = [
    "How many days do I have to return an item?",
    "Can I return shoes I have worn outside?",
    "Do I pay for return shipping when I return an item for a refund?",
    "Can I get a refund for a sale item?",
    "How long does standard shipping take within the US?",
    "Is standard shipping within the US free on orders over $75?",
    "If I order before 2 p.m. Eastern Time on a business day, does it ship the same day?",
    "Who pays import duties on orders shipped to the European Union?",
    "Which payment methods do you accept?",
    "What currency am I charged in if I order from Canada?",
    "Can I combine two gift cards in one order?",
    "How long is the warranty on shoes?",
    "Does the warranty cover worn-down tread on hiking shoes?",
    "The zipper on my jacket broke 2 months after delivery. Can I still return it for a refund?",
    "How do I reset my password?",
    "Can I add a guest order to a new account?",
    "How many Trail Club points do I earn per dollar?",
    "When will I get a tracking number?",
    "How long do I have to cancel an order after placing it?",
    "What should I do if my package is marked delivered but I cannot find it?",
]

Seed the cache with the answers of our RAG to these questions.

In [11]:
import uuid
from datetime import datetime, timezone


def upsert(question: str, answer: str, source_doc_ids: list[str]) -> None:
    client.upsert(
        CACHE_COLLECTION,
        points=[
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=embed(question),
                payload={
                    "question": question,
                    "answer": answer,
                    "source_doc_ids": source_doc_ids,
                    # Optional: if cached answers never expire, leave this field out,
                    # and the expiry filter in cache_lookup() too.
                    "expires_at": (datetime.now(timezone.utc) + CACHE_TTL).isoformat(),
                },
            )
        ],
        wait=True,
    )


for question in SEED_QUESTIONS:
    result = rag_answer(question)
    upsert(question, result["answer"], result["source_doc_ids"])

print(client.count(CACHE_COLLECTION, exact=True).count)

20


## Set Up Cache Lookup

Without a threshold, a lookup always returns the closest cached question. With one, a question that scores under it gets nothing back, so it goes to RAG.

In [12]:
def cache_lookup(
    question: str, threshold: float | None = None
) -> models.ScoredPoint | None:
    points = client.query_points(
        CACHE_COLLECTION,
        query=embed(question),
        # Optional: skip cached answers that have expired.
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="expires_at",
                    range=models.DatetimeRange(gt=datetime.now(timezone.utc)),
                )
            ]
        ),
        limit=1,
        score_threshold=threshold,
    ).points
    return points[0] if points else None

In [13]:
def show_closest(questions: list[str]) -> None:
    for question in questions:
        match = cache_lookup(question)
        print(f"{match.score:.4f}  {question}")
        print(f"        closest: {match.payload['question']}")


show_closest(
    [
        "Can I get a refund on a sale item?",
        "Who pays the import duties on orders shipped to the European Union?",
        "When is my card charged for an order?",
    ]
)

0.9877  Can I get a refund on a sale item?
        closest: Can I get a refund for a sale item?


0.9922  Who pays the import duties on orders shipped to the European Union?
        closest: Who pays import duties on orders shipped to the European Union?


0.5605  When is my card charged for an order?
        closest: What currency am I charged in if I order from Canada?


### Calibrate the Threshold

We pick the threshold on a synthetic golden set: incoming questions, each labeled with whether the cache should answer it. The dataset keeps a test half; we split its other half into train and val by cached question, so that rewordings of one question never land in different parts.

In [14]:
import json
import urllib.request
from pathlib import Path

DATASET_URL = "https://raw.githubusercontent.com/qdrant/examples/master/semantic-cache/semantic_cache_dataset.json"
DATASET_FILE = Path("semantic_cache_dataset.json")

if not DATASET_FILE.exists():
    urllib.request.urlretrieve(DATASET_URL, DATASET_FILE)
dataset = json.loads(DATASET_FILE.read_text())

# Each group of the golden set is built around one of the cached questions.
assert [group["seed"] for group in dataset["groups"]] == SEED_QUESTIONS

In [15]:
from itertools import cycle


class Row(TypedDict):
    text: str
    should_hit: bool
    # The cached question this question should hit; None for unrelated questions.
    cached_question: str | None
    part: str
    score: float
    closest: str


def add_row(
    text: str, should_hit: bool, cached_question: str | None, part: str
) -> None:
    closest = cache_lookup(text)  # one lookup per question, without a threshold
    rows.append(
        {
            "text": text,
            "should_hit": should_hit,
            "cached_question": cached_question,
            "part": part,
            "score": closest.score,
            "closest": closest.payload["question"],
        }
    )


# Keep the dataset's test half; alternate its tune half between train and val by cached question.
rows: list[Row] = []
tune_parts = cycle(["train", "val"])
for group in dataset["groups"]:
    part = "test" if group["split"] == "test" else next(tune_parts)
    for query in group["queries"]:
        add_row(query["text"], query["label"] == "hit", group["seed"], part)

tune_parts = cycle(["train", "val"])
for query in dataset["unrelated"]:
    part = "test" if query["split"] == "test" else next(tune_parts)
    add_row(query["text"], False, None, part)

train, val, test = (
    [row for row in rows if row["part"] == p] for p in ("train", "val", "test")
)
for name, part_rows in (("train", train), ("val", val), ("test", test)):
    should_hit = sum(row["should_hit"] for row in part_rows)
    print(
        f"{name:5} {should_hit} should hit, {len(part_rows) - should_hit} should miss"
    )

train 11 should hit, 8 should miss
val   10 should hit, 7 should miss
test  21 should hit, 15 should miss


Sweep the threshold: pick it on train, check it on val, pick again on train and val together if val shows wrong hits, and measure it once on test.

A **correct hit** serves the answer of the question's own cached question. A **wrong hit** serves any other answer: to a question that should miss, or the answer of a different cached question.

In [16]:
from collections.abc import Callable


def is_correct_hit(row: Row) -> bool:
    # Correct only if the cache serves the answer of this question's own cached question.
    return row["should_hit"] and row["closest"] == row["cached_question"]


def hits(
    part_rows: list[Row], value: Callable[[Row], float], threshold: float
) -> tuple[int, int]:
    """Correct and wrong hits when the cache serves every question with value >= threshold."""
    served = [row for row in part_rows if value(row) >= threshold]
    correct = sum(is_correct_hit(row) for row in served)
    return correct, len(served) - correct


def pick(
    part_rows: list[Row], value: Callable[[Row], float], candidates: list[float]
) -> float:
    """The candidate with the most correct hits and zero wrong hits; the middle one if several tie."""
    results = {t: hits(part_rows, value, t) for t in candidates}
    safe = [t for t in candidates if results[t][1] == 0]
    best = max(results[t][0] for t in safe)
    tied = [t for t in safe if results[t][0] == best]
    return tied[(len(tied) - 1) // 2]


def report(
    step: str, part_rows: list[Row], value: Callable[[Row], float], threshold: float
) -> None:
    correct, wrong = hits(part_rows, value, threshold)
    should_hit = sum(row["should_hit"] for row in part_rows)
    print(f"{step:26} {threshold:.2f}  correct {correct}/{should_hit}  wrong {wrong}")


def score(row: Row) -> float:
    return row["score"]


CANDIDATES = [i / 100 for i in range(50, 100)]

picked_on_train = pick(train, score, CANDIDATES)
report("Pick on train", train, score, picked_on_train)
report("Check on val", val, score, picked_on_train)

picked = pick(train + val, score, CANDIDATES)
report("Pick again on train + val", train + val, score, picked)
report("Measure once on test", test, score, picked)

Pick on train              0.86  correct 7/11  wrong 0
Check on val               0.86  correct 5/10  wrong 2
Pick again on train + val  0.98  correct 7/21  wrong 0
Measure once on test       0.98  correct 7/21  wrong 0


Why train alone wasn't enough: the questions val caught.

In [17]:
for row in val:
    if row["score"] >= picked_on_train and not is_correct_hit(row):
        print(f"{row['score']:.4f}  {row['text']}")
        print(f"        closest: {row['closest']}")

0.8973  Can I return shoes I have only worn indoors?
        closest: Can I return shoes I have worn outside?
0.9796  Is standard shipping within the US free on orders under $75?
        closest: Is standard shipping within the US free on orders over $75?


A lower threshold serves more questions, and more of them wrongly:

In [18]:
for threshold in (0.98, 0.95, 0.90, 0.80, 0.70):
    report("train + val", train + val, score, threshold)

train + val                0.98  correct 7/21  wrong 0
train + val                0.95  correct 10/21  wrong 1
train + val                0.90  correct 11/21  wrong 1
train + val                0.80  correct 13/21  wrong 4
train + val                0.70  correct 16/21  wrong 4


### Hard Hits vs Threshold

A paraphrase that should hit, and a one-word change that should miss:

In [19]:
show_closest(
    [
        "Is there a fee to send something back if I want my money back?",
        "If I order after 2 p.m. Eastern Time on a business day, does it ship the same day?",
    ]
)

0.6802  Is there a fee to send something back if I want my money back?
        closest: Do I pay for return shipping when I return an item for a refund?


0.9724  If I order after 2 p.m. Eastern Time on a business day, does it ship the same day?
        closest: If I order before 2 p.m. Eastern Time on a business day, does it ship the same day?


### Optional: Resolve Hard Hits with Jev

Keep the sure threshold, and add a gray zone under it where [Jev](https://docs.typesafe.ai/introduction), a TypeSafe model, double-checks the cache. Jev sees only the two questions, never the cached answer, and returns the probability that the answer to the cached question also answers the new one.

We call Jev through the OpenRouter Python client and pin the model version used for these results.

In [20]:
JEV_MODEL = "typesafe/jev-1.13-20260917"
JEV_INSTRUCTIONS = (
    "A support assistant cached an answer to cached_question. "
    "Would that same answer fully and correctly answer new_question? "
    "Answer yes only if both questions ask for the same information under the same conditions."
)


class JevCheck(TypedDict):
    probability: float
    latency_s: float
    cost_usd: float


def jev_probability(cached_question: str, new_question: str) -> JevCheck:
    start = time.perf_counter()

    # The decisions endpoint is in alpha on OpenRouter, so its request format can change.
    response = openrouter.alpha.decisions.create(
        model=JEV_MODEL,
        state={"cached_question": cached_question, "new_question": new_question},
        questions={"same_answer": {"type": "noul", "instructions": JEV_INSTRUCTIONS}},
    )
    answer = response.answers.get("same_answer")
    if answer is None:
        raise RuntimeError(f"Jev returned no answer to the question: {response}")

    return {
        "probability": answer.noul,
        "latency_s": time.perf_counter() - start,
        "cost_usd": response.usage.cost,
    }


check = jev_probability("How do I reset my password?", "How can I reset my password?")
print(f"{check['probability']:.2f}")

0.96


Pick the two new numbers the same way as the main threshold: on train, check them on val, pick again on train and val, and measure the whole pipeline once on test.

- The gray zone starts at the lowest score of a should-hit question against its own cached question, rounded down.
- The Jev threshold comes from the same sweep, run on the questions in the gray zone.

In [21]:
import math

SURE_THRESHOLD = picked
jev_cache: dict[tuple[str, str], JevCheck] = {}


def jev_for(row: Row) -> float:
    # Ask Jev once per question, however many candidate thresholds we try.
    key = (row["closest"], row["text"])
    if key not in jev_cache:
        jev_cache[key] = jev_probability(*key)
    return jev_cache[key]["probability"]


def with_jev(gray_zone: float) -> Callable[[Row], float]:
    """Sure scores always pass, scores under the gray zone never do, Jev decides in between."""

    def value(row: Row) -> float:
        if row["score"] >= SURE_THRESHOLD:
            return 1.0
        if row["score"] < gray_zone:
            return -1.0
        return jev_for(row)

    return value


def fit(part_rows: list[Row]) -> tuple[float, float]:
    # The gray zone starts at the lowest score of a correct hit, rounded down.
    gray_zone = (
        math.floor(min(row["score"] for row in part_rows if is_correct_hit(row)) * 100)
        / 100
    )
    return gray_zone, pick(
        part_rows, with_jev(gray_zone), [i / 100 for i in range(101)]
    )


gray_zone, classifier = fit(train)
print(f"Picked on train: gray zone from {gray_zone:.2f}")
report("Check on val", val, with_jev(gray_zone), classifier)

GRAY_ZONE_THRESHOLD, CLASSIFIER_THRESHOLD = fit(train + val)
print(f"Picked on train + val: gray zone from {GRAY_ZONE_THRESHOLD:.2f}")
report(
    "Measure once on test", test, with_jev(GRAY_ZONE_THRESHOLD), CLASSIFIER_THRESHOLD
)
report("Cosine only, on test", test, score, SURE_THRESHOLD)

reached_jev = [
    row for row in test if GRAY_ZONE_THRESHOLD <= row["score"] < SURE_THRESHOLD
]
print(f"{len(reached_jev)} of {len(test)} test questions reached Jev")

Picked on train: gray zone from 0.56


Check on val               0.21  correct 9/10  wrong 0


Picked on train + val: gray zone from 0.47


Measure once on test       0.29  correct 21/21  wrong 0
Cosine only, on test       0.98  correct 7/21  wrong 0
24 of 36 test questions reached Jev


What Jev says on the two hard cases:

In [22]:
for cached_question, new_question in [
    (
        "Do I pay for return shipping when I return an item for a refund?",
        "Is there a fee to send something back if I want my money back?",
    ),
    (
        "If I order before 2 p.m. Eastern Time on a business day, does it ship the same day?",
        "If I order after 2 p.m. Eastern Time on a business day, does it ship the same day?",
    ),
]:
    check = jev_probability(cached_question, new_question)
    decision = (
        "serve" if check["probability"] >= CLASSIFIER_THRESHOLD else "send to RAG"
    )
    print(f"{check['probability']:.2f}  {decision:11}  {new_question}")

0.91  serve        Is there a fee to send something back if I want my money back?


0.04  send to RAG  If I order after 2 p.m. Eastern Time on a business day, does it ship the same day?


The trade-off: every question in the gray zone pays for a Jev call. Compare its cost and latency with a RAG call on your own pipeline.

In [23]:
import statistics

calls = list(jev_cache.values())
print(
    f"{len(calls)} Jev calls: median {statistics.median(c['latency_s'] for c in calls):.2f} s, "
    f"mean ${statistics.mean(c['cost_usd'] for c in calls):.7f} per call"
)

49 Jev calls: median 0.27 s, mean $0.0000145 per call


## Serve from the Cache

All of it together: a cache lookup first, RAG on a miss, and the new answer stored in the cache. The cost shown is the LLM cost; embedding the question adds a fraction of a cent on top.

In [24]:
THRESHOLD = picked


class Served(TypedDict):
    answer: str
    hit: bool
    latency_s: float
    cost_usd: float


def ask(question: str) -> Served:
    start = time.perf_counter()

    # With the optional Jev check, look up with GRAY_ZONE_THRESHOLD and send scores under THRESHOLD to Jev.
    match = cache_lookup(question, THRESHOLD)
    if match:
        return {
            "answer": match.payload["answer"],
            "hit": True,
            "latency_s": time.perf_counter() - start,
            # No LLM call; the optional Jev check would add its cost here.
            "cost_usd": 0.0,
        }

    result = rag_answer(question)
    # Store the new answer, so the next similar question is a cache hit.
    upsert(question, result["answer"], result["source_doc_ids"])
    return {
        "answer": result["answer"],
        "hit": False,
        "latency_s": time.perf_counter() - start,
        "cost_usd": result["cost_usd"],
    }

Ask a question the cache hasn't seen, then the same question in other words:

In [25]:
for question in [
    "When is my card charged for an order?",
    "When is my card charged for my order?",
]:
    result = ask(question)
    print(
        "cache hit " if result["hit"] else "cache miss",
        f"{result['latency_s']:.2f} s",
        f"${result['cost_usd']:.6f}",
    )
    print("    ", result["answer"])

cache miss 3.45 s $0.000044
     Your card is charged when your order ships, not when you place it.


cache hit  0.57 s $0.000000
     Your card is charged when your order ships, not when you place it.


### Serve with the Jev Check (Optional)

The same `ask()`, with the Jev check for scores in the gray zone.

In [26]:
def ask_with_jev(question: str) -> Served:
    start = time.perf_counter()

    match = cache_lookup(question, GRAY_ZONE_THRESHOLD)
    jev_cost = 0.0
    if match and match.score < THRESHOLD:
        check = jev_probability(match.payload["question"], question)
        jev_cost = check["cost_usd"]
        if check["probability"] < CLASSIFIER_THRESHOLD:
            match = None

    if match:
        return {
            "answer": match.payload["answer"],
            "hit": True,
            "latency_s": time.perf_counter() - start,
            "cost_usd": jev_cost,
        }

    result = rag_answer(question)
    upsert(question, result["answer"], result["source_doc_ids"])
    return {
        "answer": result["answer"],
        "hit": False,
        "latency_s": time.perf_counter() - start,
        "cost_usd": jev_cost + result["cost_usd"],
    }


for question in [
    "Is there a fee to send something back if I want my money back?",
    "If I order after 2 p.m. Eastern Time on a business day, does it ship the same day?",
]:
    result = ask_with_jev(question)
    print(
        "cache hit " if result["hit"] else "cache miss",
        f"{result['latency_s']:.2f} s",
        f"${result['cost_usd']:.6f}",
    )
    print("    ", result["answer"])

cache hit  0.80 s $0.000015
     Yes. A $6.95 return shipping fee is deducted from your refund.


cache miss 3.46 s $0.000065
     No. Orders placed after 2 p.m. Eastern Time on a business day ship the next business day.


## Keep Cached Answers Fresh

A cached answer goes stale when it gets old, or when the document behind it changes. When a document changes, update it in the knowledge base, then delete the cached answers built from its old version, filtering on `source_doc_ids`.

For old answers: the lookup already skips answers past `expires_at`, and `delete_expired()` frees their space. Run it periodically, for example from a scheduled job.

In [27]:
def delete_expired() -> None:
    client.delete(
        CACHE_COLLECTION,
        points_selector=models.Filter(
            must=[
                models.FieldCondition(
                    key="expires_at",
                    range=models.DatetimeRange(lte=datetime.now(timezone.utc)),
                )
            ]
        ),
        wait=True,
    )


# Pretend the cached answer to one question is older than its lifetime.
question = "How do I reset my password?"
cached = cache_lookup(question)
client.set_payload(
    CACHE_COLLECTION,
    payload={
        "expires_at": (datetime.now(timezone.utc) - timedelta(seconds=1)).isoformat()
    },
    points=[cached.id],
    wait=True,
)

print(f"{cached.score:.4f}  closest before expiry: {cached.payload['question']}")
after = cache_lookup(question)
print(f"{after.score:.4f}  closest after expiry:  {after.payload['question']}")

print(
    client.count(CACHE_COLLECTION, exact=True).count, "entries before delete_expired()"
)
delete_expired()
print(client.count(CACHE_COLLECTION, exact=True).count, "entries after")

1.0000  closest before expiry: How do I reset my password?


0.2549  closest after expiry:  What should I do if my package is marked delivered but I cannot find it?
22 entries before delete_expired()
21 entries after


## Clean Up

In [28]:
client.delete_collection(CACHE_COLLECTION)
client.delete_collection(KNOWLEDGE_BASE_COLLECTION)

True